# Assignment 05 - Feature Engineering & Regression Modeling
**Dataset:** `student_dataset_dirty.csv`

This notebook covers:
- Part A: Data Cleaning
- Part B: Feature Engineering
- Part C: Linear Regression (predict `Total_Score`)
- Part D: Logistic Regression (predict `Result`)

> This is a **basic starter template**. Fill in the TODOs, add your own analysis, plots, and explanations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score, ConfusionMatrixDisplay
)

pd.set_option('display.max_columns', None)
%matplotlib inline

## Part A — Data Cleaning & Preparation

In [ ]:
df = pd.read_csv('student_dataset_dirty.csv')
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe(include='all')

In [ ]:
# Missing values per column
df.isna().sum()

In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

In [ ]:
# Handle outliers / invalid ranges (basic capping example — adjust as needed)
df.loc[(df['Age'] < 5) | (df['Age'] > 25), 'Age'] = np.nan
df.loc[(df['Attendance_Percentage'] < 0) | (df['Attendance_Percentage'] > 100), 'Attendance_Percentage'] = np.nan
df.loc[(df['Study_Hours_per_day'] < 0) | (df['Study_Hours_per_day'] > 16), 'Study_Hours_per_day'] = np.nan

for c in ['Math_Score', 'Science_Score', 'English_Score']:
    df.loc[(df[c] < 0) | (df[c] > 100), c] = np.nan

df[num_cols].describe()

In [ ]:
# Handle missing values
# Numeric -> median imputation
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

# Categorical -> mode imputation
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

df.isna().sum()

In [ ]:
# Recompute Total_Score consistently from the three subject scores
df['Total_Score'] = df['Math_Score'] + df['Science_Score'] + df['English_Score']
df[['Math_Score','Science_Score','English_Score','Total_Score']].head()

## Part B — Feature Engineering

In [ ]:
# Encode categorical variables
encode_cols = ['Gender', 'City', 'Class', 'Parent_Education', 'Internet_Access', 'Attendance_Category']

df_encoded = pd.get_dummies(df, columns=encode_cols, drop_first=True)
df_encoded.head()

## Part C — Linear Regression
**Target:** `Total_Score`

Note: We exclude `Math_Score`, `Science_Score`, `English_Score` since `Total_Score` is directly derived from them (would cause data leakage / trivial prediction).

In [ ]:
drop_for_linear = ['Student_ID', 'Name', 'Result', 'Math_Score', 'Science_Score', 'English_Score', 'Total_Score']
X_lin = df_encoded.drop(columns=drop_for_linear, errors='ignore')
y_lin = df_encoded['Total_Score']

X_train, X_test, y_train, y_test = train_test_split(X_lin, y_lin, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lin_model = LinearRegression()
lin_model.fit(X_train_scaled, y_train)
y_pred = lin_model.predict(X_test_scaled)

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Total_Score')
plt.ylabel('Predicted Total_Score')
plt.title('Actual vs Predicted (Linear Regression)')
plt.show()

In [ ]:
# Coefficient interpretation
coef_df = pd.DataFrame({'Feature': X_lin.columns, 'Coefficient': lin_model.coef_})
coef_df.sort_values('Coefficient', ascending=False)

## Part D — Logistic Regression
**Target:** `Result` (pass = 1, fail = 0)

In [ ]:
df_encoded['Result_binary'] = (df['Result'] == 'pass').astype(int)

drop_for_log = ['Student_ID', 'Name', 'Result', 'Result_binary']
X_log = df_encoded.drop(columns=drop_for_log, errors='ignore')
y_log = df_encoded['Result_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X_log, y_log, test_size=0.2, random_state=42, stratify=y_log
)

scaler2 = StandardScaler()
X_train_scaled = scaler2.fit_transform(X_train)
X_test_scaled = scaler2.transform(X_test)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)
y_pred = log_model.predict(X_test_scaled)
y_prob = log_model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Fail','Pass']).plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, y_prob):.2f}')
plt.plot([0,1], [0,1], 'r--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
# Class balance check
df['Result'].value_counts(normalize=True)